# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR\^2 dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset metadata (Croissant schema) is:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

---

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and sample records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n\n{metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview
Let's review the available record sets in this dataset, and examine their fields.

Entities in Croissant are referenced by their `@id`. We'll display those below.

In [ ]:
# List record sets and their @ids
if getattr(metadata, 'record_sets', None):
    record_sets = metadata.record_sets
else:
    # `recordSet` field may be non-standard (lowercase s), so try to extract that
    record_sets = getattr(metadata, 'recordSet', [])
    if hasattr(record_sets, 'to_json'):
        # If record_sets is an mlcroissant object, convert to list
        record_sets = [record_sets]

if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Record sets available:")
    for rs in record_sets:
        print(f"- @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)}")

# In practice, also try reading via the mlcroissant API directly:
available_record_sets = list(dataset.record_sets())
for rs in available_record_sets:
    print(f"Available record set: {rs['@id']}")

# For each, print its fields by @id
print("\nFields in each record set (by @id):\n")
for rs in available_record_sets:
    print(f"Record set @id: {rs['@id']}")
    fields = rs.get('field', [])
    # field can be a dict, a list, or missing
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict) and '@id' in field:
            print(f"  - {field['@id']}")
        elif isinstance(field, str):
            print(f"  - {field}")
    print()

## 3. Data Extraction
Load data from the primary record set (by its `@id`) into a pandas DataFrame for analysis.

**Note:** Use the actual `@id` values taken from the overview step above.

In [ ]:
# Identify record set IDs
record_set_ids = [rs['@id'] for rs in available_record_sets]
print(f'Record set IDs found: {record_set_ids}')

# Example: Select the main clinical data record set for data extraction; 
# It's typical for the largest tabular file/object to be named along 'clinical records', 'main', or dataset name.
if record_set_ids:
    main_record_set_id = record_set_ids[0]  # Use the first one as an example
else:
    raise RuntimeError('No record sets found.')

# Extract dataframes for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

df = dataframes[main_record_set_id]
print(f"Fields in main record set ({main_record_set_id}):")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (by its `@id`), filter on a threshold, normalize it, and group by a categorical attribute.

Replace the variables below by referencing the correct column name (which should correspond to the field `@id`).

In [ ]:
# Choose field IDs (replace below with actual ones from the overview)
# For demonstration, select as if the columns appear as field '@id' values from schema
numeric_field_id = None
group_field_id = None

# Try guessing field ids for demonstration from dataframe columns
for col in df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'gender' in col.lower() or 'location' in col.lower() or 'msi' in col.lower():
        group_field_id = col

if not numeric_field_id:
    numeric_field_id = df.columns[0]
if not group_field_id:
    group_field_id = df.columns[-1]

print(f"Numeric field ID: {numeric_field_id}")
print(f"Group field ID: {group_field_id}")

# Cast numeric field if possible
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove records where the numeric field is missing
filtered_df = df[df[numeric_field_id].notnull()]
# Set a filter threshold (for example, age > 50)
threshold = 50
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping: mean (or count if categorical) by group_field_id
if group_field_id in filtered_df.columns:
    grouped_summary = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_summary.head())

## 5. Visualization

Visualize the distribution of the chosen numeric field and show its distribution by category group.

_Requires matplotlib or seaborn._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# Boxplot by category
plt.figure(figsize=(8,4))
sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
plt.title(f"{numeric_field_id} grouped by {group_field_id}")
plt.xticks(rotation=45)
plt.show()

## 6. Conclusion

- The dataset provides clinicopathological features of second primary colorectal cancers in survivors. You have loaded its metadata, identified recordsets and fields by their Croissant `@id`, and demonstrated simple exploratory analyses by field `@id`.
- Analyses can be extended to survival, anatomical, and biomarker status stratifications.

Explore the schema for further fields and leverage Croissant IDs to ensure future-proof, robust referencing throughout your analysis pipeline.